# Cortical representations of languages during natural dialogue

## pycortex

In [33]:
# library
import cortex
import nibabel
import numpy as np
import numexpr
import sys
import os

from cortex import freesurfer
cortex.options.usercfg


# subject
subj='sub-OSU01'
LLM='CHATGPTNEOX'
xfmName=subj


# directroy 
datadir=os.getcwd()+'../../derivative/preprocessed/{subj}/'
savedir=os.getcwd()+'../../results/'
refEPI=datadir+'target.nii'


In [14]:
### 2. freesurfer import

# freesurfer.import_subj(subj)
# cortex.align.automatic(subj, xfmName, refEPI, use_fs_bbr=True)
# freesurfer.import_flat(subj, 'full')

In [32]:

def getniidata(niifile):
    nii=nibabel.load(niifile)
    epi=nii.get_data().astype(float).squeeze()
    data=np.transpose(epi,(2,1,0))
    
    return data

def getniidata_zero2nan(niifile):
    nii=nibabel.load(niifile)
    epi=nii.get_data().astype(float).squeeze()
    epi=np.where(epi== 0, np.nan, epi)
    data=np.transpose(epi,(2,1,0))
    
    return data
    
    
# Prediction_performance
indresultdir=f'{savedir}{subj}/'
prefix=f'{indresultdir}RidgeResult_{LLM}_{subj}_'


## same-prediction model (Figure 1d)
file_main=f'same_FDRcorr_mean.nii'
color_main="hot"
sc_max_main=0.6

## cross-prediction model (Figure 2b)
file_cross=f'cross_FDRcorr_mean.nii'

d_main=getniidata_zero2nan(prefix+file_main)
d_cross=getniidata_zero2nan(prefix+file_cross)

dv_main=cortex.Volume(d_main, subj, xfmName, cmap=color_main, vmin=0, vmax=sc_max_main)
dv_cross=cortex.Volume(d_cross, subj, xfmName, cmap=color_main, vmin=0, vmax=sc_max_main)



# Variance partitioning
## using the full model, 1d (Extended Data Figure 3)
A='Prod'
B='Comp'
postfix='.nii'

color_vp1d="inferno"
sc_max_vp1d=0.2
sc_min_vp1d=0

### large language model
d_vpmain_prod=f'{prefix}{LLM}_{A}_UniqVP{postfix}'
d_vpmain_comp=f'{prefix}{LLM}_{B}_UniqVP{postfix}'
tmp_d_vpmain_prod=getniidata(d_vpmain_prod)
tmp_d_vpmain_comp=getniidata(d_vpmain_comp)
d_vpmain_prod=cortex.Volume(tmp_d_vpmain_prod, subj, xfmName, cmap=color_vp1d, vmin=sc_min_vp1d, vmax=sc_max_vp1d)
d_vpmain_comp=cortex.Volume(tmp_d_vpmain_comp, subj, xfmName, cmap=color_vp1d, vmin=sc_min_vp1d, vmax=sc_max_vp1d)

### part-of-speech
d_vpos_prod=f'{prefix}POS_{A}_UniqVP{postfix}'
d_vpos_comp=f'{prefix}POS_{B}_UniqVP{postfix}'
tmp_d_vpos_prod=getniidata(d_vpos_prod)
tmp_d_vpos_comp=getniidata(d_vpos_comp)
d_vpos_prod=cortex.Volume(tmp_d_vpos_prod, subj, xfmName, cmap=color_vp1d, vmin=sc_min_vp1d, vmax=sc_max_vp1d)
d_vpos_comp=cortex.Volume(tmp_d_vpos_comp, subj, xfmName, cmap=color_vp1d, vmin=sc_min_vp1d, vmax=sc_max_vp1d)

### morpheme count
d_vpmr_prod=f'{prefix}MRPL_{A}_UniqVP{postfix}'
d_vpmr_comp=f'{prefix}MRPL_{B}_UniqVP{postfix}'
tmp_d_vpmr_prod=getniidata(d_vpmr_prod)
tmp_d_vpmr_comp=getniidata(d_vpmr_comp)
d_vpmr_prod=cortex.Volume(tmp_d_vpmr_prod, subj, xfmName, cmap=color_vp1d, vmin=sc_min_vp1d, vmax=sc_max_vp1d)
d_vpmr_comp=cortex.Volume(tmp_d_vpmr_comp, subj, xfmName, cmap=color_vp1d, vmin=sc_min_vp1d, vmax=sc_max_vp1d)

### syllable count
d_vpsy_prod=f'{prefix}SYL_{A}_UniqVP{postfix}'
d_vpsy_comp=f'{prefix}SYL_{B}_UniqVP{postfix}'
tmp_d_vpsy_prod=getniidata(d_vpsy_prod)
tmp_d_vpsy_comp=getniidata(d_vpsy_comp)
d_vpsy_prod=cortex.Volume(tmp_d_vpsy_prod, subj, xfmName, cmap=color_vp1d, vmin=sc_min_vp1d, vmax=sc_max_vp1d)
d_vpsy_comp=cortex.Volume(tmp_d_vpsy_comp, subj, xfmName, cmap=color_vp1d, vmin=sc_min_vp1d, vmax=sc_max_vp1d)

### cochleargram
d_vpcg_prod=f'{prefix}CCG_{A}_UniqVP{postfix}'
d_vpcg_comp=f'{prefix}CCG_{B}_UniqVP{postfix}'
tmp_d_vpcg_prod=getniidata(d_vpcg_prod)
tmp_d_vpcg_comp=getniidata(d_vpcg_comp)
d_vpcg_prod=cortex.Volume(tmp_d_vpcg_prod, subj, xfmName, cmap=color_vp1d, vmin=sc_min_vp1d, vmax=sc_max_vp1d)
d_vpcg_comp=cortex.Volume(tmp_d_vpcg_comp, subj, xfmName, cmap=color_vp1d, vmin=sc_min_vp1d, vmax=sc_max_vp1d)

### motion
d_vpmt=f'{prefix}MOT_{A}_UniqVP{postfix}'
tmp_d_vpmt=getniidata(d_vpmt)
d_vpmt=cortex.Volume(tmp_d_vpmt, subj, xfmName, cmap=color_vp1d, vmin=sc_min_vp1d, vmax=sc_max_vp1d)


## using the full model, 2d (Extended Data Figure 3,4)
color_vp2d='RdBu_covar'
sc_max_vp2d=0.2
sc_min_vp2d=0

d_vpmain2d=cortex.dataset.Volume2D(tmp_d_vpmain_prod, tmp_d_vpmain_comp, subj, xfmName, vmin=sc_min_vp2d, vmax=sc_max_vp2d, vmin2=sc_min_vp2d, vmax2=sc_max_vp2d, cmap=color_vp2d)
d_vpos2d  =cortex.dataset.Volume2D(tmp_d_vpos_prod,   tmp_d_vpos_comp,   subj, xfmName, vmin=sc_min_vp2d, vmax=sc_max_vp2d, vmin2=sc_min_vp2d, vmax2=sc_max_vp2d, cmap=color_vp2d)
d_vpmr2d  =cortex.dataset.Volume2D(tmp_d_vpmr_prod,   tmp_d_vpmr_comp,   subj, xfmName, vmin=sc_min_vp2d, vmax=sc_max_vp2d, vmin2=sc_min_vp2d, vmax2=sc_max_vp2d, cmap=color_vp2d)
d_vpsy2d  =cortex.dataset.Volume2D(tmp_d_vpsy_prod,   tmp_d_vpsy_comp,   subj, xfmName, vmin=sc_min_vp2d, vmax=sc_max_vp2d, vmin2=sc_min_vp2d, vmax2=sc_max_vp2d, cmap=color_vp2d)
d_vpcg2d  =cortex.dataset.Volume2D(tmp_d_vpcg_prod,   tmp_d_vpcg_comp,   subj, xfmName, vmin=sc_min_vp2d, vmax=sc_max_vp2d, vmin2=sc_min_vp2d, vmax2=sc_max_vp2d, cmap=color_vp2d)


## using the reduced model, 2d (Figure 3a)
color_vp2d='RdBu_covar'
sc_max_vp2d=0.2
sc_min_vp2d=0

d_vpmain2_prod=f'{prefix}SingleFeature_UniqVP_{A}{postfix}'
d_vpmain2_comp=f'{prefix}SingleFeature_UniqVP_{B}{postfix}'
tmp_d_vpmain2_prod=getniidata(d_vpmain2_prod)
tmp_d_vpmain2_comp=getniidata(d_vpmain2_comp)
d_vpmain2_2d=cortex.dataset.Volume2D(tmp_d_vpmain2_prod, tmp_d_vpmain2_comp, subj, xfmName, vmin=sc_min_vp2d, vmax=sc_max_vp2d, vmin2=sc_min_vp2d, vmax2=sc_max_vp2d, cmap=color_vp2d)



## Best variance partition RGB (Figure 3b, Extended Data Figure 5)
d_vpintsctrgb=f'{prefix}VarPart_RGB.nii'
tmp_d_vpintsctrgb=getniidata_zero2nan(d_vpintsctrgb)
d_vpintsctrgb=cortex.Volume(tmp_d_vpintsctrgb, subj, xfmName, cmap='brg', vmin=1, vmax=3)



# Weight correlation (Figure 4, Extended Data Figure 6, 7)
color_wc="jet"
sc_max_vp1d=0.6
d_wc_fdr=f'WeightCorrelation_{LLM}_{subj}_FDRcorr.nii'

d_wc_fdr=getniidata(indresultdir+d_wc_fdr)
d_wc_fdr_p=np.where(d_wc_fdr< 0, np.nan, d_wc_fdr)
d_wc_fdr_n=np.where(d_wc_fdr>= 0, np.nan, d_wc_fdr)

d_wc_fdr_p=cortex.Volume(d_wc_fdr_p, subj, xfmName, cmap=color_wc, vmin=-sc_max_vp1d, vmax=sc_max_vp1d)
d_wc_fdr_n=cortex.Volume(d_wc_fdr_n, subj, xfmName, cmap=color_wc, vmin=-sc_max_vp1d, vmax=sc_max_vp1d)



# PCA 
## RGB (PC1, PC2, PC3; Figure 5i,j, Extended Data Figure 8, 9)
d_pcs_r=f'PCA_Result_{LLM}_{subj}_{A}_RGBmap_R.nii'
d_pcs_g=f'PCA_Result_{LLM}_{subj}_{A}_RGBmap_G.nii'
d_pcs_b=f'PCA_Result_{LLM}_{subj}_{A}_RGBmap_B.nii'
d_pco_r=f'PCA_Result_{LLM}_{subj}_{B}_RGBmap_R.nii'
d_pco_g=f'PCA_Result_{LLM}_{subj}_{B}_RGBmap_G.nii'
d_pco_b=f'PCA_Result_{LLM}_{subj}_{B}_RGBmap_B.nii'

d_pcs_r=getniidata(indresultdir+d_pcs_r)
d_pcs_g=getniidata(indresultdir+d_pcs_g)
d_pcs_b=getniidata(indresultdir+d_pcs_b)
d_pco_r=getniidata(indresultdir+d_pco_r)
d_pco_g=getniidata(indresultdir+d_pco_g)
d_pco_b=getniidata(indresultdir+d_pco_b)

alpha=np.ones(d_pcs_r.shape)
alpha[np.isnan(d_pcs_r)]=0

pcrgb_prod=cortex.VolumeRGB(d_pcs_r, d_pcs_g, d_pcs_b, subj, xfmName, alpha=alpha, description='pc_prod')
pcrgb_comp=cortex.VolumeRGB(d_pco_r, d_pco_g, d_pco_b, subj, xfmName, alpha=alpha, description='pc_comp')


## Each PC (Extended Data Figure 10)
color_pc1d="RdBu_r"
sc_max_pc1d=1.6

d_pc1_prod=f'PCA_Result_{LLM}_{subj}_{A}_pc1.nii'
d_pc2_prod=f'PCA_Result_{LLM}_{subj}_{A}_pc2.nii'
d_pc3_prod=f'PCA_Result_{LLM}_{subj}_{A}_pc3.nii'
d_pc4_prod=f'PCA_Result_{LLM}_{subj}_{A}_pc4.nii'

d_pc1_comp=f'PCA_Result_{LLM}_{subj}_{B}_pc1.nii'
d_pc2_comp=f'PCA_Result_{LLM}_{subj}_{B}_pc2.nii'
d_pc3_comp=f'PCA_Result_{LLM}_{subj}_{B}_pc3.nii'
d_pc4_comp=f'PCA_Result_{LLM}_{subj}_{B}_pc4.nii'
d_pc5_comp=f'PCA_Result_{LLM}_{subj}_{B}_pc5.nii'
d_pc6_comp=f'PCA_Result_{LLM}_{subj}_{B}_pc6.nii'

tmp_d_pc1_prod=getniidata(indresultdir+d_pc1_prod)
tmp_d_pc2_prod=getniidata(indresultdir+d_pc2_prod)
tmp_d_pc3_prod=getniidata(indresultdir+d_pc3_prod)
tmp_d_pc4_prod=getniidata(indresultdir+d_pc4_prod)

tmp_d_pc1_comp=getniidata(indresultdir+d_pc1_comp)
tmp_d_pc2_comp=getniidata(indresultdir+d_pc2_comp)
tmp_d_pc3_comp=getniidata(indresultdir+d_pc3_comp)
tmp_d_pc4_comp=getniidata(indresultdir+d_pc4_comp)
tmp_d_pc5_comp=getniidata(indresultdir+d_pc5_comp)
tmp_d_pc6_comp=getniidata(indresultdir+d_pc6_comp)

d_pc1_prod=cortex.Volume(tmp_d_pc1_prod, subj, xfmName, cmap=color_pc1d, vmin=-sc_max_pc1d, vmax=sc_max_pc1d)
d_pc2_prod=cortex.Volume(tmp_d_pc2_prod, subj, xfmName, cmap=color_pc1d, vmin=-sc_max_pc1d, vmax=sc_max_pc1d)
d_pc3_prod=cortex.Volume(tmp_d_pc3_prod, subj, xfmName, cmap=color_pc1d, vmin=-sc_max_pc1d, vmax=sc_max_pc1d)
d_pc4_prod=cortex.Volume(tmp_d_pc4_prod, subj, xfmName, cmap=color_pc1d, vmin=-sc_max_pc1d, vmax=sc_max_pc1d)

d_pc1_comp=cortex.Volume(tmp_d_pc1_comp, subj, xfmName, cmap=color_pc1d, vmin=-sc_max_pc1d, vmax=sc_max_pc1d)
d_pc2_comp=cortex.Volume(tmp_d_pc2_comp, subj, xfmName, cmap=color_pc1d, vmin=-sc_max_pc1d, vmax=sc_max_pc1d)
d_pc3_comp=cortex.Volume(tmp_d_pc3_comp, subj, xfmName, cmap=color_pc1d, vmin=-sc_max_pc1d, vmax=sc_max_pc1d)
d_pc4_comp=cortex.Volume(tmp_d_pc4_comp, subj, xfmName, cmap=color_pc1d, vmin=-sc_max_pc1d, vmax=sc_max_pc1d)
d_pc5_comp=cortex.Volume(tmp_d_pc5_comp, subj, xfmName, cmap=color_pc1d, vmin=-sc_max_pc1d, vmax=sc_max_pc1d)
d_pc6_comp=cortex.Volume(tmp_d_pc6_comp, subj, xfmName, cmap=color_pc1d, vmin=-sc_max_pc1d, vmax=sc_max_pc1d)


volumes = {
           # Figure 1,2, Extended Data Figure 1,2
           'fig1_Pred_same': dv_main, 'fig2_Pred_cross': dv_cross,
    
           # Extended Data Figure 3
           'exfig3_VPLLM': d_vpmain2d, 'exfig3_VPpos': d_vpos2d, 'exfig3_VPmr': d_vpmr2d, 'exfig3_VPsy': d_vpsy2d, 'exfig3_VPcg': d_vpcg2d, 
           'exfig3_VPmt': d_vpmt,
           
           # Figure 3a, Extended Data Figure 4
           'fig3a_VPreduced': d_vpmain2_2d,
    
           # Figure 3b, Extended Data Figure 5
           'fig3b_BestVP': d_vpintsctrgb,
    
           # Figure 4a,b, Extended Data Figure 6, 7
           'fig4a_WeightCorr_p': d_wc_fdr_p, 'fig4b_WeightCorr_n': d_wc_fdr_n,
    
           # Figure 5i,j, Extended Data Figure 8, 9
           'fig5i_PCA_prod': pcrgb_prod, 'fig5j_PCA_comp': pcrgb_comp,
        
           # Extended Data Figure 10
           'exfig10_PC1_prod': d_pc1_prod, 'exfig10_PC2_prod': d_pc2_prod, 'exfig10_PC3_prod': d_pc3_prod, 'exfig10_PC4_prod': d_pc4_prod, 
           'exfig10_PC1_comp': d_pc1_comp, 'exfig10_PC2_comp': d_pc2_comp, 'exfig10_PC3_comp': d_pc3_comp,
           'exfig10_PC4_comp': d_pc4_comp, 'exfig10_PC5_comp': d_pc5_comp, 'exfig10_PC6_comp': d_pc6_comp,           
          
          }

im=cortex.webshow(volumes, title=subj)





/Users/ymm/opt/anaconda3/lib/python3.7/site-packages/ipykernel_launcher.py:10: DeprecationWarning: get_data() is deprecated in favor of get_fdata(), which has a more predictable return type. To obtain get_data() behavior going forward, use numpy.asanyarray(img.dataobj).

* deprecated from version: 3.0
* Will raise <class 'nibabel.deprecator.ExpiredDeprecationError'> as of version: 5.0
  # Remove the CWD from sys.path while we load stuff.
/Users/ymm/opt/anaconda3/lib/python3.7/site-packages/ipykernel_launcher.py:3: DeprecationWarning: get_data() is deprecated in favor of get_fdata(), which has a more predictable return type. To obtain get_data() behavior going forward, use numpy.asanyarray(img.dataobj).

* deprecated from version: 3.0
* Will raise <class 'nibabel.deprecator.ExpiredDeprecationError'> as of version: 5.0
  This is separate from the ipykernel package so we can avoid doing imports until


Started server on port 64704
Stopping server
